# Machine Learning Based Student Academic Performance Prediction and Risk Analysis System

**B.Tech Computer Science mini project | Jupyter Notebook edition**

This notebook is the guided ML-learning version of the project. Its companion React + Flask website is in `student-performance-system/`. Read `README.md` for quick start or `PROJECT_GUIDE_HINGLISH.md` for the easy end-to-end explanation.

Work through dataset generation/import, validation and preprocessing, the five syllabus models, correct regression/classification evaluation, charts, interactive analytics, a reusable prediction function, and model export.

> **Demo-data warning:** the included 600-row dataset is synthetic demonstration data, not real student records. Predictions are estimates, not guaranteed outcomes, and must not replace teacher judgment.

## Notebook contents
1. Setup and configuration
2. Generate or load data
3. Validate and preprocess
4. Train the five required models
5. Evaluate regression and classification separately
6. Compare algorithms and inspect actual-vs-predicted results
7. View static charts and interactively explore academic features
8. Predict for a student and review recommendations
9. Export sample data and save trained pipelines
10. Project summary, limitations, and run instructions

**Target-label thresholds:** performance is Good at `>= 75`, Average at `>= 50 and < 75`, otherwise Poor. Risk is Low at `>= 70`, Medium at `>= 45 and < 70`, otherwise High.

## 1. Setup and configuration

**Fastest Windows setup:** go back to the repository folder and double-click `RUN_NOTEBOOK.bat`. It installs `requirements-notebook.txt` for the current Windows user and opens this notebook in JupyterLab. The launcher starts JupyterLab as a Python module, so it does not depend on the global `jupyter-lab` command being available on PATH.

**Manual package setup:** from the repository folder, run `python -m pip install --user -r requirements-notebook.txt`. Then open this notebook in VS Code with the Jupyter extension, select the Python interpreter where you installed the requirements, and choose **Run All**. For JupyterLab startup, use `RUN_NOTEBOOK.bat`.

This notebook uses Python, pandas/NumPy, scikit-learn, Matplotlib, Plotly, and ipywidgets. If you use an external CSV, place it at `dataset/student_performance.csv` before running the dataset cell. Otherwise the notebook creates a reproducible 600-row synthetic sample.

> Do not put real student records, passwords, or API keys in this public project. The example records are synthetic.

In [ ]:
# If installing manually from the repository root, run: %pip install -r requirements-notebook.txt
from pathlib import Path
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, mean_absolute_error,
    mean_squared_error, precision_score, r2_score, recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

RANDOM_STATE = 42
N_SAMPLES = 600
DATA_DIR = Path('dataset')
MODEL_DIR = Path('saved_models')
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_CSV_PATH = DATA_DIR / 'student_performance_sample.csv'
EXTERNAL_CSV_PATH = DATA_DIR / 'student_performance.csv'  # Optional: place your CSV here
FEATURES = ['attendance', 'previous_marks', 'study_hours', 'internal_marks']
NUMERIC_COLUMNS = FEATURES + ['final_marks']
PERFORMANCE_LABELS = ['Poor Performance', 'Average Performance', 'Good Performance']
RISK_LABELS = ['High Risk', 'Medium Risk', 'Low Risk']
print('Setup complete. Working directory:', Path.cwd().resolve())

## 2. Generate or load the dataset

By default the notebook creates 600 reproducible synthetic records with bounded, correlated academic features. To use an external CSV, place it at `dataset/student_performance.csv` or change `EXTERNAL_CSV_PATH`. Required numeric columns are attendance (0–100), previous_marks (0–100), study_hours (0–24), internal_marks (0–20), and final_marks (0–100). `student_id` and `student_name` are optional; labels are always derived from final marks using the documented thresholds so inconsistent uploaded labels cannot silently become targets.

In [ ]:
def performance_category(marks):
    if marks >= 75:
        return 'Good Performance'
    if marks >= 50:
        return 'Average Performance'
    return 'Poor Performance'


def academic_risk(marks):
    if marks >= 70:
        return 'Low Risk'
    if marks >= 45:
        return 'Medium Risk'
    return 'High Risk'


def make_synthetic_dataset(n_samples=600, random_state=42):
    rng = np.random.default_rng(random_state)
    attendance = np.clip(rng.normal(76, 14, n_samples), 10, 100)
    previous = np.clip(30 + 0.62 * attendance + rng.normal(0, 12, n_samples), 0, 100)
    study = np.clip(rng.gamma(shape=2.2, scale=1.35, size=n_samples) + attendance * 0.018, 0.25, 12)
    internal = np.clip(0.10 * previous + 0.055 * attendance + rng.normal(0, 2.4, n_samples), 0, 20)
    final = np.clip(
        0.18 * attendance + 0.32 * previous + 0.15 * (study * 8.0)
        + 0.25 * (internal * 5.0) + 0.15 * (study ** 2) + 8.0
        + rng.normal(0, 7.0, n_samples), 0, 100
    )
    frame = pd.DataFrame({
        'student_id': [f'STU2024{i:04d}' for i in range(1, n_samples + 1)],
        'student_name': [f'Student_{i}' for i in range(1, n_samples + 1)],
        'attendance': attendance.round(1),
        'previous_marks': previous.round(1),
        'study_hours': study.round(1),
        'internal_marks': internal.round(1),
        'final_marks': final.round(1),
    })
    frame['performance_category'] = frame['final_marks'].map(performance_category)
    frame['academic_risk'] = frame['final_marks'].map(academic_risk)
    return frame


if EXTERNAL_CSV_PATH.exists():
    raw_df = pd.read_csv(EXTERNAL_CSV_PATH)
    dataset_source = f'External CSV: {EXTERNAL_CSV_PATH}'
else:
    raw_df = make_synthetic_dataset(N_SAMPLES, RANDOM_STATE)
    dataset_source = 'Synthetic demonstration data'
    raw_df.to_csv(SAMPLE_CSV_PATH, index=False)

print(dataset_source, '| rows:', len(raw_df))
display(raw_df.head())

## 3. Validate, clean, and prepare data

Exact duplicate rows are removed. Numeric missing values are imputed using medians learned from the training partition only; scaling likewise lives inside model pipelines and is fitted only during training. Rows with missing target marks are excluded because a supervised target cannot be imputed safely. Invalid out-of-range values cause an explicit error. Student identifiers/names and final marks are never predictor features.

In [ ]:
REQUIRED_COLUMNS = set(NUMERIC_COLUMNS)
missing_columns = sorted(REQUIRED_COLUMNS - set(raw_df.columns))
if missing_columns:
    raise ValueError(f'CSV is missing required columns: {missing_columns}')

df = raw_df.copy().drop_duplicates().reset_index(drop=True)
for column in NUMERIC_COLUMNS:
    df[column] = pd.to_numeric(df[column], errors='coerce')
df = df.dropna(subset=['final_marks']).reset_index(drop=True)
if len(df) < 20:
    raise ValueError('At least 20 usable rows with final_marks are required for a train/test split.')

ranges = {
    'attendance': (0, 100), 'previous_marks': (0, 100),
    'study_hours': (0, 24), 'internal_marks': (0, 20), 'final_marks': (0, 100),
}
for column, (lower, upper) in ranges.items():
    invalid = df[column].notna() & ~df[column].between(lower, upper)
    if invalid.any():
        raise ValueError(f'{column} must be in [{lower}, {upper}]; found {int(invalid.sum())} invalid row(s).')

if 'student_id' not in df:
    df['student_id'] = [f'CSV{i:05d}' for i in range(1, len(df) + 1)]
if 'student_name' not in df:
    df['student_name'] = df['student_id'].astype(str)
df['performance_category'] = df['final_marks'].map(performance_category)
df['academic_risk'] = df['final_marks'].map(academic_risk)

train_idx, test_idx = train_test_split(
    df.index, test_size=0.20, random_state=RANDOM_STATE
)
train_df, test_df = df.loc[train_idx].copy(), df.loc[test_idx].copy()
print(f'Rows after duplicate removal: {len(df)} | train: {len(train_df)} | test: {len(test_df)}')
print('Missing numeric feature values (imputed within each training pipeline):')
display(df[FEATURES].isna().sum().to_frame('missing values'))
print('Derived target distributions:')
display(df[['performance_category', 'academic_risk']].apply(pd.Series.value_counts).fillna(0).astype(int))
display(df[NUMERIC_COLUMNS].describe().round(2))

## 4. Train the five syllabus models

**Regression models:** (1) simple Linear Regression uses only `previous_marks`; (2) Multiple Linear Regression uses all four input features; (3) Polynomial Regression applies degree-2 polynomial feature expansion followed by Linear Regression. **Classifiers:** KNN predicts performance category; Logistic Regression predicts academic risk. All imputation/scaling steps are in pipelines and therefore fitted on training data only. Final marks are used only as regression targets and to derive classification labels, never as an input feature.

In [ ]:
simple_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LinearRegression()),
])
multiple_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LinearRegression()),
])
polynomial_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('polynomial_features', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('model', LinearRegression()),
])
knn_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier(n_neighbors=7)),
])
logistic_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])

simple_model.fit(train_df[['previous_marks']], train_df['final_marks'])
multiple_model.fit(train_df[FEATURES], train_df['final_marks'])
polynomial_model.fit(train_df[FEATURES], train_df['final_marks'])
knn_model.fit(train_df[FEATURES], train_df['performance_category'])
logistic_model.fit(train_df[FEATURES], train_df['academic_risk'])

models = {
    'Linear Regression (single feature)': simple_model,
    'Multiple Linear Regression': multiple_model,
    'Polynomial Regression (degree 2)': polynomial_model,
    'K-Nearest Neighbors': knn_model,
    'Logistic Regression': logistic_model,
}
print('All five required models trained successfully.')

## 5. Evaluate model performance

Regression and classification are evaluated separately on the held-out test set. Regression reports MAE, MSE, RMSE, and R². Classifiers report accuracy, weighted precision/recall/F1 and confusion matrices. Values below are calculated from the data and are not hardcoded.

In [ ]:
regression_specs = {
    'Linear Regression (single feature)': (simple_model, ['previous_marks']),
    'Multiple Linear Regression': (multiple_model, FEATURES),
    'Polynomial Regression (degree 2)': (polynomial_model, FEATURES),
}
regression_results = {}
for name, (model, columns) in regression_specs.items():
    actual = test_df['final_marks'].to_numpy()
    predicted = model.predict(test_df[columns])
    regression_results[name] = {
        'MAE': mean_absolute_error(actual, predicted),
        'MSE': mean_squared_error(actual, predicted),
        'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
        'R2': r2_score(actual, predicted),
        'actual': actual,
        'predicted': predicted,
    }
regression_table = pd.DataFrame({
    name: {metric: values[metric] for metric in ['MAE', 'MSE', 'RMSE', 'R2']}
    for name, values in regression_results.items()
}).T
display(regression_table.round(3).style.format('{:.3f}').background_gradient(cmap='Blues', subset=['R2']))

In [ ]:
classification_specs = {
    'K-Nearest Neighbors': (knn_model, 'performance_category', PERFORMANCE_LABELS),
    'Logistic Regression': (logistic_model, 'academic_risk', RISK_LABELS),
}
classification_results = {}
for name, (model, target_column, labels) in classification_specs.items():
    actual = test_df[target_column].to_numpy()
    predicted = model.predict(test_df[FEATURES])
    classification_results[name] = {
        'Accuracy': accuracy_score(actual, predicted),
        'Precision (weighted)': precision_score(actual, predicted, labels=labels, average='weighted', zero_division=0),
        'Recall (weighted)': recall_score(actual, predicted, labels=labels, average='weighted', zero_division=0),
        'F1 (weighted)': f1_score(actual, predicted, labels=labels, average='weighted', zero_division=0),
        'matrix': confusion_matrix(actual, predicted, labels=labels),
        'labels': labels,
        'actual': actual,
        'predicted': predicted,
    }
classification_table = pd.DataFrame({
    name: {metric: values[metric] for metric in ['Accuracy', 'Precision (weighted)', 'Recall (weighted)', 'F1 (weighted)']}
    for name, values in classification_results.items()
}).T
display(classification_table.round(3).style.format('{:.3f}').background_gradient(cmap='Greens'))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, values) in zip(axes, classification_results.items()):
    matrix = values['matrix']
    image = ax.imshow(matrix, cmap='Blues')
    ax.set_title(f'{name} — confusion matrix')
    ax.set_xticks(range(len(values['labels'])), values['labels'], rotation=25, ha='right')
    ax.set_yticks(range(len(values['labels'])), values['labels'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    for (row, col), value in np.ndenumerate(matrix):
        ax.text(col, row, str(value), ha='center', va='center', color='black')
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## 6. Algorithm comparison and regression diagnostics

Actual-vs-predicted plots include a diagonal reference line; closer points indicate smaller prediction errors. The side-by-side summary keeps regression and classification metrics distinct because they measure different tasks.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, values) in zip(axes, regression_results.items()):
    ax.scatter(values['actual'], values['predicted'], alpha=0.65, edgecolor='none')
    lower = min(values['actual'].min(), values['predicted'].min())
    upper = max(values['actual'].max(), values['predicted'].max())
    ax.plot([lower, upper], [lower, upper], linestyle='--', color='crimson', linewidth=1)
    ax.set_title(name)
    ax.set_xlabel('Actual final marks')
    ax.set_ylabel('Predicted final marks')
    ax.grid(alpha=0.2)
plt.suptitle('Held-out test set: actual vs predicted marks')
plt.tight_layout()
plt.show()

print('Regression model comparison (lower error is better; higher R² is better):')
display(regression_table.sort_values('RMSE').round(3))
print('Classification model comparison (higher scores are better):')
display(classification_table.sort_values('F1 (weighted)', ascending=False).round(3))

## 7. Student analytics and polynomial curve

The static overview below compares each academic feature with final marks and shows the performance/risk distribution. The polynomial curve is a one-feature illustration using `previous_marks`, fitted only on the training partition.

The interactive explorer after this chart lets you choose the x-axis feature and performance group. Hover over points for student ID, name, marks, and predicted training label; click the Plotly legend to show/hide groups. These relationships are descriptive, not proof that one factor causes marks to change.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
scatter_specs = [
    ('attendance', 'Attendance (%) vs final marks'),
    ('study_hours', 'Study hours/day vs final marks'),
    ('previous_marks', 'Previous marks vs final marks'),
    ('internal_marks', 'Internal marks vs final marks'),
]
for ax, (column, title) in zip(axes.flat[:4], scatter_specs):
    ax.scatter(df[column], df['final_marks'], alpha=0.4, s=18)
    ax.set_title(title)
    ax.set_xlabel(column.replace('_', ' ').title())
    ax.set_ylabel('Final marks')
    ax.grid(alpha=0.2)

df['performance_category'].value_counts().reindex(PERFORMANCE_LABELS, fill_value=0).plot(
    kind='bar', ax=axes.flat[4], color=['#ef8354', '#f2cc8f', '#4f9d69']
)
axes.flat[4].set_title('Performance distribution')
axes.flat[4].set_ylabel('Students')
axes.flat[4].tick_params(axis='x', rotation=20)
df['academic_risk'].value_counts().reindex(RISK_LABELS, fill_value=0).plot(
    kind='bar', ax=axes.flat[5], color=['#d1495b', '#edae49', '#4f9d69']
)
axes.flat[5].set_title('Academic risk distribution')
axes.flat[5].set_ylabel('Students')
axes.flat[5].tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

# A separate one-input polynomial curve for an interpretable syllabus demonstration.
curve_x = np.linspace(train_df['previous_marks'].min(), train_df['previous_marks'].max(), 250)
curve_frame = pd.DataFrame({'previous_marks': curve_x})
curve_y = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('polynomial_features', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('model', LinearRegression()),
])
curve_y.fit(train_df[['previous_marks']], train_df['final_marks'])
plt.figure(figsize=(8, 5))
plt.scatter(train_df['previous_marks'], train_df['final_marks'], alpha=0.35, label='Training records')
plt.plot(curve_x, curve_y.predict(curve_frame), color='crimson', linewidth=2, label='Degree-2 polynomial fit')
plt.xlabel('Previous semester marks')
plt.ylabel('Final examination marks')
plt.title('Polynomial regression curve (single-feature illustration)')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

### Interactive academic analytics explorer

Choose an academic feature and performance group. Hover over any point to inspect its synthetic student details; use the chart legend to toggle performance groups.

In [ ]:
import plotly.express as px
import ipywidgets as widgets
from IPython.display import clear_output

ANALYTICS_FEATURES = {
    'Attendance (%)': 'attendance',
    'Study hours/day': 'study_hours',
    'Previous semester marks': 'previous_marks',
    'Internal examination marks': 'internal_marks',
}
ANALYTICS_COLORS = {
    'Good Performance': '#7c6cf2',
    'Average Performance': '#28a789',
    'Poor Performance': '#f0a44b',
}

feature_picker = widgets.Dropdown(
    options=list(ANALYTICS_FEATURES), value='Attendance (%)', description='X-axis:',
)
category_picker = widgets.Dropdown(
    options=['All groups'] + PERFORMANCE_LABELS, value='All groups', description='Group:',
)
chart_output = widgets.Output()

def _draw_interactive_analytics(_change=None):
    feature_label = feature_picker.value
    feature_column = ANALYTICS_FEATURES[feature_label]
    selected_group = category_picker.value
    chart_data = df if selected_group == 'All groups' else df[df['performance_category'] == selected_group]
    with chart_output:
        clear_output(wait=True)
        if chart_data.empty:
            print('There are no records in this group.')
            return
        figure = px.scatter(
            chart_data,
            x=feature_column,
            y='final_marks',
            color='performance_category',
            color_discrete_map=ANALYTICS_COLORS,
            hover_name='student_name',
            hover_data={
                'student_id': True,
                'attendance': ':.1f',
                'previous_marks': ':.1f',
                'study_hours': ':.1f',
                'internal_marks': ':.1f',
                'final_marks': ':.1f',
                'academic_risk': True,
                'performance_category': False,
            },
            labels={
                feature_column: feature_label,
                'final_marks': 'Final marks (0?100)',
                'performance_category': 'Performance group',
                'academic_risk': 'Academic risk',
                'student_id': 'Student ID',
                'attendance': 'Attendance (%)',
                'previous_marks': 'Previous marks',
                'study_hours': 'Study hours/day',
                'internal_marks': 'Internal marks',
            },
            title=f'{feature_label} vs final marks ? {len(chart_data)} students',
        )
        figure.update_traces(marker={'size': 9, 'opacity': 0.78, 'line': {'width': 0.4, 'color': 'white'}})
        figure.update_layout(
            template='plotly_white',
            height=490,
            legend_title_text='Performance group',
            margin={'l': 48, 'r': 20, 't': 62, 'b': 50},
            hoverlabel={'namelength': 0},
        )
        figure.update_yaxes(range=[0, 100], gridcolor='#e7e8f0', zeroline=False)
        figure.update_xaxes(gridcolor='#e7e8f0', zeroline=False)
        display(figure)

feature_picker.observe(_draw_interactive_analytics, names='value')
category_picker.observe(_draw_interactive_analytics, names='value')
display(widgets.HBox([feature_picker, category_picker]))
display(chart_output)
_draw_interactive_analytics()

## 8. Predict for a student

The prediction function checks all required input fields and their ranges. It uses Multiple Linear Regression for the expected final-mark estimate, KNN for performance class, and Logistic Regression for risk class. Attendance/study observations and recommendations are transparent threshold-based guidance, not additional machine-learning models. To change the chosen predictive model, explicitly select another trained model rather than silently blending outputs.

In [ ]:
def predict_student(student_id, student_name, attendance, previous_marks, study_hours, internal_marks):
    values = {
        'attendance': attendance,
        'previous_marks': previous_marks,
        'study_hours': study_hours,
        'internal_marks': internal_marks,
    }
    limits = {
        'attendance': (0, 100), 'previous_marks': (0, 100),
        'study_hours': (0, 24), 'internal_marks': (0, 20),
    }
    for field, value in values.items():
        if not isinstance(value, (int, float, np.integer, np.floating)) or not np.isfinite(value):
            raise ValueError(f'{field} must be a finite number.')
        lower, upper = limits[field]
        if not lower <= float(value) <= upper:
            raise ValueError(f'{field} must be between {lower} and {upper}.')
    if not str(student_id).strip() or not str(student_name).strip():
        raise ValueError('Student ID and student name cannot be empty.')

    features = pd.DataFrame([{key: float(value) for key, value in values.items()}], columns=FEATURES)
    estimated_marks = float(multiple_model.predict(features)[0])
    estimated_marks = float(np.clip(estimated_marks, 0, 100))
    predicted_performance = str(knn_model.predict(features)[0])
    predicted_risk = str(logistic_model.predict(features)[0])

    if attendance < 60:
        attendance_analysis = 'Attendance is low; regular class participation may help.'
    elif attendance < 75:
        attendance_analysis = 'Attendance is moderate; improving consistency is recommended.'
    else:
        attendance_analysis = 'Attendance is good; maintain this consistency.'
    if study_hours < 1:
        study_analysis = 'Reported study time is very low; establish a daily routine.'
    elif study_hours < 3:
        study_analysis = 'A consistent study schedule with focused practice may help.'
    else:
        study_analysis = 'Reported study time is positive; continue focused revision.'

    recommendations = []
    if attendance < 75:
        recommendations.append('Set an attendance improvement goal and discuss barriers with a teacher.')
    if previous_marks < 50:
        recommendations.append('Review foundational topics and schedule weekly practice with feedback.')
    if study_hours < 2:
        recommendations.append('Start with a realistic daily study plan and short, distraction-free sessions.')
    if internal_marks < 10:
        recommendations.append('Seek support on internal-assessment topics and clarify upcoming deadlines.')
    if not recommendations:
        recommendations.append('Maintain current habits and use regular formative assessments to monitor progress.')

    return {
        'student_id': str(student_id).strip(),
        'student_name': str(student_name).strip(),
        'predicted_final_marks': round(estimated_marks, 1),
        'performance_category': predicted_performance,
        'academic_risk': predicted_risk,
        'attendance_analysis': attendance_analysis,
        'study_habit_analysis': study_analysis,
        'recommendations': recommendations,
        'disclaimer': 'This is an estimate based on the demonstration/training data, not a guaranteed result. Use teacher judgment and verified student information.',
    }


example_prediction = predict_student(
    student_id='DEMO001', student_name='Demo Student',
    attendance=78, previous_marks=68, study_hours=2.5, internal_marks=14,
)
display(pd.Series({key: value for key, value in example_prediction.items() if key != 'recommendations'}))
print('Recommendations:')
for recommendation in example_prediction['recommendations']:
    print('-', recommendation)

### Optional interactive prediction form

If `ipywidgets` is installed and enabled in Jupyter, use this form. Otherwise, call `predict_student(...)` directly in a code cell as shown above.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import clear_output

    student_id_widget = widgets.Text(value='DEMO002', description='Student ID:')
    student_name_widget = widgets.Text(value='Sample Student', description='Name:')
    attendance_widget = widgets.FloatSlider(value=75, min=0, max=100, step=1, description='Attendance:')
    previous_widget = widgets.FloatSlider(value=65, min=0, max=100, step=1, description='Previous:')
    study_widget = widgets.FloatSlider(value=2.5, min=0, max=24, step=0.5, description='Hours/day:')
    internal_widget = widgets.FloatSlider(value=13, min=0, max=20, step=0.5, description='Internal:')
    predict_button = widgets.Button(description='Predict performance', button_style='primary')
    prediction_output = widgets.Output()

    def _on_predict(_):
        with prediction_output:
            clear_output(wait=True)
            try:
                result = predict_student(
                    student_id_widget.value, student_name_widget.value,
                    attendance_widget.value, previous_widget.value,
                    study_widget.value, internal_widget.value,
                )
                display(pd.Series({k: v for k, v in result.items() if k != 'recommendations'}))
                print('Recommendations:')
                for item in result['recommendations']:
                    print('-', item)
            except ValueError as exc:
                print(f'Input error: {exc}')

    predict_button.on_click(_on_predict)
    display(widgets.VBox([
        student_id_widget, student_name_widget, attendance_widget, previous_widget,
        study_widget, internal_widget, predict_button, prediction_output,
    ]))
except ImportError:
    print('ipywidgets is not installed. Use the predict_student(...) function directly.')

## 9. Export sample CSV and save trained models

The sample CSV is provided for external upload/demo. Trained pipelines include their preprocessing steps and can be reloaded with joblib. Pickle/joblib files should only be loaded from trusted sources.

In [ ]:
sample_df = make_synthetic_dataset(N_SAMPLES, RANDOM_STATE)
sample_df.to_csv(SAMPLE_CSV_PATH, index=False)

artifact_paths = {}
for filename, model in {
    'linear_regression_single_feature.joblib': simple_model,
    'multiple_linear_regression.joblib': multiple_model,
    'polynomial_regression_degree2.joblib': polynomial_model,
    'knn_performance_classifier.joblib': knn_model,
    'logistic_risk_classifier.joblib': logistic_model,
}.items():
    destination = MODEL_DIR / filename
    joblib.dump(model, destination)
    artifact_paths[filename] = str(destination.resolve())

metadata = {
    'project': 'Machine Learning Based Student Academic Performance Prediction and Risk Analysis System',
    'dataset_source': dataset_source,
    'synthetic_demo_data': dataset_source == 'Synthetic demonstration data',
    'training_rows': int(len(train_df)),
    'test_rows': int(len(test_df)),
    'features': FEATURES,
    'performance_thresholds': {'good_min': 75, 'average_min': 50},
    'risk_thresholds': {'low_min': 70, 'medium_min': 45},
    'regression_metrics': regression_table.round(6).to_dict(orient='index'),
    'classification_metrics': classification_table.round(6).to_dict(orient='index'),
    'artifacts': artifact_paths,
}
metadata_path = MODEL_DIR / 'model_metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

print('Sample CSV saved to:', SAMPLE_CSV_PATH.resolve())
print('Model artifacts saved to:', MODEL_DIR.resolve())
print('Metadata saved to:', metadata_path.resolve())
print('Rows in generated sample:', len(sample_df))

## 10. Project summary, limitations, and run instructions

- **Exactly five predictive algorithms:** single-feature Linear Regression; Multiple Linear Regression; degree-2 Polynomial Regression (PolynomialFeatures + Linear Regression); KNN; and Logistic Regression. No other predictive algorithm is used.
- **Labels:** performance and risk groups are derived from final marks using the thresholds stated at the top. These demonstration labels are not a validated educational risk standard.
- **Leakage control:** final marks are never included in the predictor matrix. Imputation and scaling are fitted inside training pipelines; evaluation uses a held-out test partition.
- **Interactive analytics:** Plotly shows each academic feature against final marks; widgets filter by feature and performance group. Hover reveals synthetic student details. Correlation is not causation.
- **Synthetic-data limitation:** generated correlations do not represent actual students or prove real-world model performance. Use authorized, representative, appropriately governed data and validate on future/unseen cohorts before drawing conclusions.
- **Run:** from the repository root, double-click `RUN_NOTEBOOK.bat`, open this notebook, and select **Run All**. For VS Code, install `requirements-notebook.txt`, choose that Python kernel, then run all cells.
- **External dataset:** place an authorized CSV at `dataset/student_performance.csv` before the data-loading cell. Otherwise the notebook generates the 600-row sample at `dataset/student_performance_sample.csv`.
- **Generated files:** sample CSV and model pipelines/metadata are written to `dataset/` and `saved_models/`; these are local outputs and are intentionally excluded from Git.

Predictions are estimates, not guarantees. Never use this prototype by itself to grade, punish, or make decisions about a student.